In [5]:
import scanpy as sc
import os
import glob
import pandas as pd
import anndata as ad
from sklearn.metrics import silhouette_score
import seaborn as sns   

In [6]:
neighbors_list   = [15,10,15,30,45]
resolution_list  = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0,1.2,1.4,1.6,1.8,2.0]
OUTPUT_H5AD = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/modelo_prueba_final_fast_convergence/adata/mofa_adata_30f.h5ad'
adata_mofa = ad.read(OUTPUT_H5AD)

# 1. Cargar datos
save_dir = "/mnt/lustre/scratch/nlsas/home/ulc/co/mao/modelo_prueba_final_fast_convergence/leiden_results"
os.makedirs(save_dir, exist_ok=True)

# 2. Bucle optimizado
for k in neighbors_list:
    print(f"\n===== Procesando Vecinos (k={k}) =====")
    
    # A. Calcular vecinos y UMAP UNA SOLA VEZ por cada 'k'
    # Estos pasos son los más lentos, así que los sacamos del bucle de resoluciones
    sc.pp.neighbors(adata_mofa, n_neighbors=k, metric="cosine", use_rep='X')
    sc.tl.umap(adata_mofa) # El UMAP depende de la estructura de vecinos, no de la resolución
    
    for r in resolution_list:
        print(f"   -> Calculando Leiden (r={r})...")

        # B. Clustering Leiden (esto es rápido)
        leiden_key = f"leiden_k{k}_r{r}"
        sc.tl.leiden(adata_mofa, resolution=r, key_added=leiden_key)

        # C. Guardar AnnData ligero
        # IMPORTANTE: Añadimos 'plate' (o la columna que uses para colorear) a la lista de obs
        result_adata = sc.AnnData(
            X=None, 
            obs=adata_mofa.obs.copy(), 
            obsm={'X_umap': adata_mofa.obsm['X_umap'].copy(),
            'X_mofa': adata_mofa.X.copy()}
        )
        
        filename = f"adata_k{k}_r{r}.h5ad"
        result_adata.write(os.path.join(save_dir, filename))
        
        # Limpieza de memoria local
        del result_adata
        
        # Eliminamos la columna de leiden del objeto global para no acumular basura
        del adata_mofa.obs[leiden_key]

/mnt/netapp2/Store_uni/home/ulc/co/mao/SOFTWARE/miniconda3/envs/sc_2/lib/python3.11/site-packages/anndata/__init__.py:42: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(



===== Procesando Vecinos (k=15) =====
   -> Calculando Leiden (r=0.1)...
   -> Calculando Leiden (r=0.2)...
   -> Calculando Leiden (r=0.3)...
   -> Calculando Leiden (r=0.4)...
   -> Calculando Leiden (r=0.5)...
   -> Calculando Leiden (r=0.6)...
   -> Calculando Leiden (r=0.7)...
   -> Calculando Leiden (r=0.8)...
   -> Calculando Leiden (r=0.9)...
   -> Calculando Leiden (r=1.0)...
   -> Calculando Leiden (r=1.2)...
   -> Calculando Leiden (r=1.4)...
   -> Calculando Leiden (r=1.6)...
   -> Calculando Leiden (r=1.8)...
   -> Calculando Leiden (r=2.0)...

===== Procesando Vecinos (k=10) =====


   -> Calculando Leiden (r=0.1)...
   -> Calculando Leiden (r=0.2)...
   -> Calculando Leiden (r=0.3)...
   -> Calculando Leiden (r=0.4)...
   -> Calculando Leiden (r=0.5)...
   -> Calculando Leiden (r=0.6)...
   -> Calculando Leiden (r=0.7)...
   -> Calculando Leiden (r=0.8)...
   -> Calculando Leiden (r=0.9)...
   -> Calculando Leiden (r=1.0)...
   -> Calculando Leiden (r=1.2)...
   -> Calculando Leiden (r=1.4)...
   -> Calculando Leiden (r=1.6)...
   -> Calculando Leiden (r=1.8)...
   -> Calculando Leiden (r=2.0)...

===== Procesando Vecinos (k=15) =====
   -> Calculando Leiden (r=0.1)...
   -> Calculando Leiden (r=0.2)...
   -> Calculando Leiden (r=0.3)...
   -> Calculando Leiden (r=0.4)...
   -> Calculando Leiden (r=0.5)...
   -> Calculando Leiden (r=0.6)...
   -> Calculando Leiden (r=0.7)...
   -> Calculando Leiden (r=0.8)...
   -> Calculando Leiden (r=0.9)...
   -> Calculando Leiden (r=1.0)...
   -> Calculando Leiden (r=1.2)...
   -> Calculando Leiden (r=1.4)...
   -> Calculando

In [7]:
import scanpy as sc
import os
import glob
import pandas as pd
from sklearn.metrics import silhouette_score

# --- CONFIGURACIÓN ---
files = sorted(glob.glob(os.path.join(save_dir, "adata_k*_r*.h5ad")))

results = []

for f in files:
    try:
        basename = os.path.basename(f).replace(".h5ad", "")
        
        # 1. Leer el archivo
        adata_temp = sc.read_h5ad(f)
        
        # 2. Encontrar la columna del clustering (Leiden)
        leiden_keys = [col for col in adata_temp.obs.columns if 'leiden' in col]
        if not leiden_keys:
            continue
        leiden_key = leiden_keys[0]
        labels = adata_temp.obs[leiden_key]
        
        # Si solo hay 1 cluster, no se puede calcular Silhouette
        if len(labels.unique()) <= 1:
            print(f"{basename:<40} | {'ERROR':<10} | {'1 Cluster'}")
            continue

        # 3. Definir la matriz sobre la que calcular la distancia
        # IDEAL: Calcular sobre los Factores MOFA (espacio latente real)
        # ALTERNATIVA: Calcular sobre UMAP (espacio visual, menos riguroso pero válido si no hay MOFA)
        if 'X_mofa' in adata_temp.obsm:
            matrix = adata_temp.obsm['X_mofa']
            metric_used = "MOFA"
        elif 'X_umap' in adata_temp.obsm:
            matrix = adata_temp.obsm['X_umap']
            metric_used = "UMAP"
        else:
            print(f"{basename} no tiene X_mofa ni X_umap")
            continue
            
        # 4. Calcular Silhouette Score
       
        score = silhouette_score(matrix, labels)
        
        # 5. Guardar resultado
        results.append({
            'filename': basename,
            'k_neighbors': basename.split('_')[1].replace('k', ''), # Extraer k
            'resolution': basename.split('_')[2].replace('r', ''),  # Extraer r
            'n_clusters': len(labels.unique()),
            'metric_source': metric_used,
            'silhouette_score': score
        })
        
        print(f"{basename:<40} | {metric_used:<10} | {score:.4f}")

    except Exception as e:
        print(f"Error en {f}: {e}")

# --- RESULTADO FINAL ---
print("=" * 80)
if len(results) > 0:
    df_results = pd.DataFrame(results)
    
    # Convertir columnas a numérico para ordenar bien
    df_results['silhouette_score'] = pd.to_numeric(df_results['silhouette_score'])
    
    # Ordenar por mejor puntuación
    df_best = df_results.sort_values(by='silhouette_score', ascending=False)
    
    print("\n GANADOR (Mejor Definición de Clusters):")
    print(df_best.head(1))
    
    print("\n Ranking Completo (Top 5):")
    print(df_best[['filename', 'n_clusters', 'silhouette_score']].head(5))
    
    # Guardar tabla
    save_dir_excel = '/home/ulc/co/mao/drug_clustering/TFM_mao'

    excel_path = os.path.join(save_dir_excel, "silhouette_ranking.xlsx")
    df_best.to_excel(excel_path, index=False)  # .to_excel() para .xlsx
    print(f"\nRanking guardado en: {excel_path}")

else:
    print("No se pudieron calcular resultados.")

adata_k10_r0.1                           | MOFA       | 0.3073
adata_k10_r0.2                           | MOFA       | 0.1353
adata_k10_r0.3                           | MOFA       | 0.0869
adata_k10_r0.4                           | MOFA       | 0.0851
adata_k10_r0.5                           | MOFA       | 0.0903
adata_k10_r0.6                           | MOFA       | 0.0459
adata_k10_r0.7                           | MOFA       | 0.0437
adata_k10_r0.8                           | MOFA       | 0.0386
adata_k10_r0.9                           | MOFA       | 0.0054
adata_k10_r1.0                           | MOFA       | -0.0074
adata_k10_r1.2                           | MOFA       | -0.0283
adata_k10_r1.4                           | MOFA       | -0.0419
adata_k10_r1.6                           | MOFA       | -0.0378
adata_k10_r1.8                           | MOFA       | -0.0427
adata_k10_r2.0                           | MOFA       | -0.0492
adata_k15_r0.1                           | MOFA  

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import scanpy as sc

# Cargar el ranking
save_dir_excel = '/home/ulc/co/mao/drug_clustering/TFM_mao'
ranking = pd.read_excel(os.path.join(save_dir_excel, "silhouette_ranking.xlsx"))

files = sorted(glob.glob(os.path.join(save_dir, "adata_k*_r*.h5ad")))


for f in files:
    nombre = os.path.basename(f).replace(".h5ad", "")
    
    if nombre in ranking['filename'].values[:45]:  
        print(f"Procesando: {nombre}")  
        
        adata = sc.read_h5ad(f)
        leiden = adata.obs.columns[-1]
        plate = adata.obs.columns[2]
        
        sc.pl.umap(adata,
                   color=leiden,
                   legend_loc='on data',
                   frameon=False,
                   title=f'UMAP - {nombre}',
                   show=False)
        
        plt.savefig(os.path.join(save_dir, f'umap_{nombre}.png'),
                    dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        
print(" Gráficos completados")

 Gráficos completados


/tmp/ipykernel_1798130/2356128451.py:17: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  if nombre in ranking['n_clusters'].values[:45]:
